Import libraries

In [1]:
import os
import pickle
import numpy as np
import scanpy as sc
import pandas as pd
import anndata as ad
from pathlib import Path
import subprocess, shlex
from tqdm import tqdm

os.environ["HF_HOME"] = "/data/rbg/users/ashyamal/huggingface_cache"

import torch
from huggingface_hub import snapshot_download

Settings

In [26]:
model_folder = "../../data/STATE"
if not os.path.exists(model_folder): 
    os.makedirs(model_folder)

repo_id = "arcinstitute/SE-600M"
target_dir = os.path.join(model_folder, "SE-600M")
exclude = ["se600m_epoch4.ckpt", "se600m_epoch4.safetensors"]

Helper Functions

In [2]:
def pickle_data(data, path): 

    with open(path, "wb") as f:
        pickle.dump(data, f)

Download models and checkpoints

In [ ]:
if os.path.exists(target_dir) and os.listdir(target_dir):
    raise SystemExit(f"'{target_dir}' already exists and is not empty.")

snapshot_download(
    repo_id=repo_id,
    repo_type="model",
    revision="main",
    local_dir=target_dir,
    local_dir_use_symlinks=False,  # copy files instead of symlinking from cache
    ignore_patterns=exclude,       # skip the big epoch-4 files
    resume_download=True,
)

print(f"Downloaded to ./{target_dir} (excluded: {', '.join(exclude)})")

Load in the data

Get the genes that are captured in state's protein embeddings

In [3]:
protein_emb_path = "../../data/STATE/SE-600M/protein_embeddings.pt"
protein_emb = torch.load(protein_emb_path)
captured_genes = list(protein_emb.keys())

In [4]:
data_folder = "../../data/TCGA/RNA"
TCGA_RNAseq_info = ad.read_h5ad(os.path.join(data_folder, "TCGA_bulk_RNA_processed_log1p.h5ad"))
data_output_path = os.path.join(data_folder, "TCGA_bulk_RNA_processed_log1p_HVG.h5ad")

N_HVG = 5000
sc.pp.highly_variable_genes(TCGA_RNAseq_info, n_top_genes=N_HVG, flavor='seurat_v3')
selected_genes = TCGA_RNAseq_info.var['highly_variable'].index[TCGA_RNAseq_info.var['highly_variable']].values.tolist()
selected_genes = [g for g in selected_genes if g in captured_genes]
print(f"There are {len(selected_genes)}")
TCGA_RNAseq_hvg = TCGA_RNAseq_info[:, selected_genes]
TCGA_RNAseq_hvg.write(data_output_path)

/data/rbg/users/ashyamal/miniconda3/envs/rna_modeling/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


There are 3678


state emb transform \
  --model-folder /data/rbg/users/ashyamal/RNA-FM-benchmarking/data/STATE/SE-600M \
  --input /data/rbg/users/ashyamal/RNA-FM-benchmarking/data/TCGA/RNA/TCGA_bulk_RNA_processed_log1p_HVG.h5ad \
  --output /data/rbg/users/ashyamal/RNA-FM-benchmarking/data/TCGA/RNA/FM_embeddings/STATE/full_tcga_STATE_emb.h5ad

Get one file per patient

In [6]:
output_folder = "../../data/TCGA/RNA/FM_embeddings/STATE/"

STATE_emb_info = ad.read_h5ad(os.path.join(output_folder, "full_tcga_STATE_emb.h5ad"))
STATE_emb = STATE_emb_info.obsm["X_state"]
patient_id = STATE_emb_info.obs["cases.submitter_id"].tolist()

for i, id_ in tqdm(enumerate(patient_id)):

    current_output_path = os.path.join(output_folder, f"{id_}.pkl")
    emb = STATE_emb[i:i+1][0]
    pickle_data(emb, current_output_path)

10478it [00:11, 916.78it/s] 
